# SleepSense — Snore Classifier Training
**Model**: EfficientNetB0 (transfer learning from ImageNet)  
**Input**: 128×128 log-mel spectrogram (3-second audio window)  
**Output**: 4-class probabilities — `snoring / breathing / silence / ambient`  
**Target**: F1 > 0.92 on held-out test set  

Run on **Google Colab** with a T4 GPU (Runtime → Change runtime type → GPU).

## 1. Environment Setup

In [ ]:
# Install dependencies (Colab only needs librosa + soundfile + opencv)
!pip install -q librosa soundfile opencv-python-headless

In [ ]:
import os, random, shutil, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import seaborn as sns

warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

print(f'TensorFlow {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 2. Constants

In [ ]:
SAMPLE_RATE  = 16_000
WINDOW_SEC   = 3.0
N_MELS       = 128
N_FFT        = 1024
HOP_LENGTH   = 512
F_MIN        = 50
F_MAX        = 8_000
IMG_SIZE     = 128
TOP_DB       = 80.0

CLASSES      = ['snoring', 'breathing', 'silence', 'ambient']
N_CLASSES    = len(CLASSES)

BATCH_SIZE   = 32
EPOCHS_HEAD  = 5    # frozen backbone
EPOCHS_FINE  = 20   # fine-tune top layers
LR_HEAD      = 1e-3
LR_FINE      = 1e-4

DATA_DIR     = '/content/data'
MODEL_DIR    = '/content/models'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

## 3. Download Datasets

In [ ]:
# ── ESC-50 (2000 clips, 50 environmental sound classes) ──────────────────────
# Class 28 = Snoring, others used as Ambient
import urllib.request

ESC50_URL = 'https://github.com/karoldvl/ESC-50/archive/master.zip'
ESC50_ZIP = os.path.join(DATA_DIR, 'esc50.zip')
ESC50_DIR = os.path.join(DATA_DIR, 'ESC-50-master')

if not os.path.exists(ESC50_DIR):
    print('Downloading ESC-50...')
    urllib.request.urlretrieve(ESC50_URL, ESC50_ZIP)
    with zipfile.ZipFile(ESC50_ZIP, 'r') as z:
        z.extractall(DATA_DIR)
    print(f'ESC-50 extracted to {ESC50_DIR}')
else:
    print('ESC-50 already downloaded')

meta = pd.read_csv(os.path.join(ESC50_DIR, 'meta', 'esc50.csv'))
print(f'ESC-50: {len(meta)} clips, {meta["target"].nunique()} classes')
print('Snoring clips:', len(meta[meta['target'] == 28]))

In [ ]:
# ── Kaggle Snoring Dataset (optional — skip if you don't have kaggle.json) ───
# If you have a Kaggle account:
#   1. Go to kaggle.com → Account → Create API Token → download kaggle.json
#   2. Upload it to Colab via the Files panel
#   3. Uncomment and run this block

# !mkdir -p ~/.kaggle && cp /content/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d tareqkhanemu/snoring -p {DATA_DIR} --unzip
# SNORING_DIR = os.path.join(DATA_DIR, 'snoring')

print('Skipping Kaggle dataset — ESC-50 snoring class will be augmented heavily.')
print('Add Kaggle dataset later to improve snoring class recall.')
SNORING_DIR = None

## 4. Build Dataset

In [ ]:
# ── Feature extraction helpers ────────────────────────────────────────────────

def load_audio(path, offset=0.0):
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True,
                        offset=offset, duration=WINDOW_SEC)
    target_len = int(SAMPLE_RATE * WINDOW_SEC)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    return y[:target_len].astype(np.float32)


def audio_to_melspec(y):
    mel = librosa.feature.melspectrogram(
        y=y, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmin=F_MIN, fmax=F_MAX
    )
    log_mel = librosa.power_to_db(mel, ref=1.0, top_db=TOP_DB)
    img = cv2.resize(log_mel, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return img.astype(np.float32)  # (128, 128)


def generate_silence(duration=WINDOW_SEC):
    rms = 10 ** (-45 / 20)
    noise = np.random.randn(int(SAMPLE_RATE * duration)).astype(np.float32)
    return noise / (noise.std() + 1e-9) * rms


def augment(y):
    if random.random() < 0.5:
        y = librosa.effects.time_stretch(y, rate=random.uniform(0.85, 1.15))
    if random.random() < 0.5:
        y = librosa.effects.pitch_shift(y, sr=SAMPLE_RATE, n_steps=random.uniform(-2, 2))
    if random.random() < 0.4:
        y = y + (np.random.randn(len(y)) * random.uniform(0.001, 0.01)).astype(np.float32)
    if random.random() < 0.3:
        y = y * (10 ** (random.uniform(-6, 6) / 20))
    target = int(SAMPLE_RATE * WINDOW_SEC)
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    return y[:target].astype(np.float32)

print('Feature extraction functions defined.')

In [ ]:
# ── Build sample list ─────────────────────────────────────────────────────────
# ESC-50 label 28 = snoring, rest = ambient
# We'll oversample snoring and add silence programmatically

records = []  # list of (path_or_None, label, augment_count)

audio_dir = os.path.join(ESC50_DIR, 'audio')

for _, row in meta.iterrows():
    fpath = os.path.join(audio_dir, row['filename'])
    if not os.path.exists(fpath):
        continue
    if row['target'] == 28:           # snoring → oversample ×5
        for _ in range(5):
            records.append((fpath, 0, True))
    elif row['target'] in {10, 11}:   # ESC-50 classes closest to breathing
        records.append((fpath, 1, True))
    else:
        records.append((fpath, 3, False))  # ambient — no augment

# Add silence samples
SILENCE_DIR = os.path.join(DATA_DIR, 'silence')
os.makedirs(SILENCE_DIR, exist_ok=True)
for i in range(400):
    sil = generate_silence()
    sp  = os.path.join(SILENCE_DIR, f'sil_{i:04d}.wav')
    sf.write(sp, sil, SAMPLE_RATE)
    records.append((sp, 2, False))

# Add Kaggle snoring dataset if available
if SNORING_DIR and os.path.exists(SNORING_DIR):
    from pathlib import Path
    for fp in Path(SNORING_DIR).rglob('*.wav'):
        lbl = 0 if 'snor' in str(fp).lower() else 1   # snoring vs breathing
        records.append((str(fp), lbl, True))
    print(f'Added Kaggle snoring dataset: {len(records)} total')

random.shuffle(records)

df = pd.DataFrame(records, columns=['path', 'label', 'do_augment'])
print(f'Total samples: {len(df)}')
print(df['label'].map({0:'snoring',1:'breathing',2:'silence',3:'ambient'}).value_counts())

In [ ]:
# ── Visualise one sample per class ───────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for cls_idx, cls_name in enumerate(CLASSES):
    sample = df[df['label'] == cls_idx].iloc[0]
    y = load_audio(sample['path'])
    spec = audio_to_melspec(y)
    axes[cls_idx].imshow(spec, origin='lower', aspect='auto', cmap='magma')
    axes[cls_idx].set_title(cls_name)
    axes[cls_idx].axis('off')
plt.suptitle('Log-mel spectrograms per class', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Pre-compute Spectrograms & Build tf.data Pipeline

In [ ]:
# Pre-compute all spectrograms to disk (avoids re-loading audio every epoch)
SPEC_DIR = os.path.join(DATA_DIR, 'spectrograms')
os.makedirs(SPEC_DIR, exist_ok=True)

spec_paths = []
labels     = []

for i, row in df.iterrows():
    try:
        y = load_audio(row['path'])
        if row['do_augment']:
            y = augment(y)
        spec = audio_to_melspec(y)
    except Exception as e:
        print(f'Skipping {row["path"]}: {e}')
        continue

    out = os.path.join(SPEC_DIR, f'{i:06d}.npy')
    np.save(out, spec)
    spec_paths.append(out)
    labels.append(row['label'])

    if i % 200 == 0:
        print(f'  {i}/{len(df)} processed...')

print(f'Done. {len(spec_paths)} spectrograms saved.')

In [ ]:
# ── Train / val / test split ──────────────────────────────────────────────────
X = np.array(spec_paths)
y = np.array(labels)

X_tmp,  X_test,  y_tmp,  y_test  = train_test_split(X, y, test_size=0.10, stratify=y, random_state=42)
X_train, X_val,  y_train, y_val  = train_test_split(X_tmp, y_tmp, test_size=0.111, stratify=y_tmp, random_state=42)
# 80% train / 10% val / 10% test

print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')
for split_name, split_y in [('train', y_train), ('val', y_val), ('test', y_test)]:
    counts = pd.Series(split_y).map({i: c for i, c in enumerate(CLASSES)}).value_counts()
    print(f'  {split_name}: {dict(counts)}')

In [ ]:
# ── tf.data pipeline ──────────────────────────────────────────────────────────

def load_spec(path, label):
    spec = tf.numpy_function(lambda p: np.load(p.numpy().decode()), [path], tf.float32)
    spec.set_shape([IMG_SIZE, IMG_SIZE])
    # EfficientNet expects (H, W, C) with 3 channels
    spec = tf.stack([spec, spec, spec], axis=-1)  # (128, 128, 3)
    label = tf.cast(label, tf.int32)
    return spec, label


def make_dataset(paths, lbls, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, lbls))
    ds = ds.map(load_spec, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(buffer_size=1000, seed=42)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_dataset(X_train, y_train, training=True)
val_ds   = make_dataset(X_val,   y_val)
test_ds  = make_dataset(X_test,  y_test)

print('tf.data pipelines ready.')

## 6. Model — EfficientNetB0 Transfer Learning

In [ ]:
def build_model(n_classes=N_CLASSES, trainable_backbone=False):
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    # Rescale [0,1] → [-1,1] as EfficientNet expects
    x = layers.Rescaling(scale=2.0, offset=-1.0)(inputs)

    backbone = keras.applications.EfficientNetB0(
        include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    backbone.trainable = trainable_backbone

    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(n_classes, activation='softmax')(x)

    return keras.Model(inputs, outputs)


model = build_model(trainable_backbone=False)
model.summary(line_length=80)

## 7. Phase 1 — Train Classification Head (backbone frozen)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(LR_HEAD),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cb_ckpt = callbacks.ModelCheckpoint(
    os.path.join(MODEL_DIR, 'best_head.keras'),
    monitor='val_accuracy', save_best_only=True, verbose=1
)
cb_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)

hist_head = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=[cb_ckpt, cb_lr]
)

plt.figure(figsize=(12, 4))
plt.subplot(1,2,1)
plt.plot(hist_head.history['accuracy'], label='train')
plt.plot(hist_head.history['val_accuracy'], label='val')
plt.title('Phase 1 — Accuracy'); plt.legend()
plt.subplot(1,2,2)
plt.plot(hist_head.history['loss'], label='train')
plt.plot(hist_head.history['val_loss'], label='val')
plt.title('Phase 1 — Loss'); plt.legend()
plt.tight_layout(); plt.show()

## 8. Phase 2 — Fine-tune (unfreeze top 30 backbone layers)

In [ ]:
# Load best head weights, then unfreeze top layers of backbone
model.load_weights(os.path.join(MODEL_DIR, 'best_head.keras'))

backbone = model.layers[3]   # EfficientNetB0 layer
backbone.trainable = True

# Freeze all except the last 30 layers
for layer in backbone.layers[:-30]:
    layer.trainable = False

trainable = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f'Trainable params: {trainable:,}')

model.compile(
    optimizer=keras.optimizers.Adam(LR_FINE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cb_ckpt2 = callbacks.ModelCheckpoint(
    os.path.join(MODEL_DIR, 'best_finetune.keras'),
    monitor='val_accuracy', save_best_only=True, verbose=1
)
cb_early = callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)
cb_cos   = callbacks.CosineDecayRestarts(
    initial_learning_rate=LR_FINE, first_decay_steps=len(X_train)//BATCH_SIZE
) if hasattr(callbacks, 'CosineDecayRestarts') else None

cbs = [cb_ckpt2, cb_early]

hist_fine = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_FINE,
    callbacks=cbs
)

plt.figure(figsize=(12, 4))
plt.subplot(1,2,1)
plt.plot(hist_fine.history['accuracy'], label='train')
plt.plot(hist_fine.history['val_accuracy'], label='val')
plt.title('Phase 2 — Accuracy'); plt.legend()
plt.subplot(1,2,2)
plt.plot(hist_fine.history['loss'], label='train')
plt.plot(hist_fine.history['val_loss'], label='val')
plt.title('Phase 2 — Loss'); plt.legend()
plt.tight_layout(); plt.show()

## 9. Evaluation

In [ ]:
model.load_weights(os.path.join(MODEL_DIR, 'best_finetune.keras'))

y_pred_proba = model.predict(test_ds)
y_pred = np.argmax(y_pred_proba, axis=1)

f1 = f1_score(y_test, y_pred, average='macro')
print(f'\nTest macro-F1: {f1:.4f}  (target > 0.92)')
print()
print(classification_report(y_test, y_pred, target_names=CLASSES))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES, yticklabels=CLASSES, cmap='Blues')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.title(f'Confusion Matrix  (F1={f1:.3f})')
plt.tight_layout(); plt.show()

if f1 < 0.90:
    print('\n⚠️  F1 below 0.90 — consider adding more snoring/breathing data or extra augmentation.')
else:
    print('\n✅  F1 target met — ready for TFLite export.')

## 10. Export to SavedModel

In [ ]:
SAVED_MODEL_PATH = os.path.join(MODEL_DIR, 'snore_classifier_savedmodel')
model.save(SAVED_MODEL_PATH)
print(f'SavedModel saved to {SAVED_MODEL_PATH}')

# Also save class mapping
import json
with open(os.path.join(MODEL_DIR, 'class_map.json'), 'w') as f:
    json.dump({i: c for i, c in enumerate(CLASSES)}, f)
print('Class map saved.')

## Done ✅
Run `02_export_tflite.ipynb` next to quantize and package the model for the mobile app.